# 🎨 Creating Images: The Magic of Diffusion Models
## From Noise to Art

**Workshop 2: Foundations of AI - Deep Learning & Computer Vision**

In this notebook, we'll explore:
- How AI creates images from pure noise
- The elegant math behind diffusion
- Why this combines SPACE + TIME understanding

In [ ]:
# =============================================================================
# PART 1: THE DIFFUSION PROCESS - ADDING NOISE
# =============================================================================
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from PIL import Image
import io
import base64
import warnings
warnings.filterwarnings('ignore')

import plotly.io as pio
pio.renderers.default = "notebook"

display(HTML("""
<div class="concept-box">
    <h2 style="margin-top:0; color:#e73c7e;">🌫️ The Core Insight: Learning to Denoise</h2>
    <p>Diffusion models learn by watching images get <strong>destroyed by noise</strong>...</p>
    <p>...then learning to <strong>reverse the process</strong>!</p>
</div>
"""))

class DiffusionProcessVisualizer:
    def __init__(self):
        self.create_sample_image()
        self.create_interface()
    
    def create_sample_image(self):
        """Create a simple recognizable pattern"""
        # Create a simple "smiley face" pattern
        img = np.zeros((32, 32))
        
        # Face circle
        y, x = np.ogrid[:32, :32]
        center = (16, 16)
        radius = 12
        mask = (x - center[0])**2 + (y - center[1])**2 <= radius**2
        img[mask] = 0.9
        
        # Eyes
        for eye_x in [11, 21]:
            eye_mask = (x - eye_x)**2 + (y - 10)**2 <= 2**2
            img[eye_mask] = 0.2
        
        # Smile
        smile_y, smile_x = np.ogrid[:32, :32]
        smile_mask = ((smile_x - 16)**2 + (smile_y - 18)**2 <= 8**2) & \
                     ((smile_x - 16)**2 + (smile_y - 18)**2 >= 5**2) & \
                     (smile_y > 18)
        img[smile_mask] = 0.2
        
        self.original_image = img
    
    def add_noise(self, img, noise_level):
        """Add Gaussian noise to image"""
        noise = np.random.randn(*img.shape) * noise_level
        noisy = img + noise
        return np.clip(noisy, 0, 1)
    
    def create_interface(self):
        self.header = widgets.HTML("""
        <div style="background: linear-gradient(-45deg, #ee7752, #e73c7e, #23a6d5, #23d5ab);
                    background-size: 400% 400%;
                    color: white; padding: 20px; border-radius: 16px; text-align: center;
                    margin-bottom: 20px; font-family: 'Inter', sans-serif;
                    box-shadow: 0 8px 30px rgba(231, 60, 126, 0.3);">
            <h3 style="margin: 0;">🌫️ Forward Diffusion: Watch an Image Dissolve into Noise</h3>
            <p style="margin: 8px 0 0 0; font-size: 14px; opacity: 0.95;">
                Drag the slider to see how noise gradually destroys the image
            </p>
        </div>
        """)
        
        self.noise_slider = widgets.FloatSlider(
            value=0, min=0, max=1, step=0.02,
            description='Noise:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='600px'),
            readout_format='.0%'
        )
        
        self.timestep_display = widgets.HTML()
        self.plot_output = widgets.Output()
        
        self.noise_slider.observe(self.update_visualization, 'value')
        
        display(self.header)
        display(widgets.HBox([self.noise_slider], layout=widgets.Layout(justify_content='center')))
        display(self.timestep_display)
        display(self.plot_output)
        
        self.update_visualization(None)
    
    def update_visualization(self, change):
        noise_level = self.noise_slider.value
        timestep = int(noise_level * 1000)
        
        # Status display
        if noise_level < 0.2:
            status = "🖼️ Clear image - recognizable"
            status_color = "#22c55e"
        elif noise_level < 0.5:
            status = "🌫️ Getting noisy - details fading"
            status_color = "#f59e0b"
        elif noise_level < 0.8:
            status = "📡 Mostly noise - hard to recognize"
            status_color = "#ef4444"
        else:
            status = "⚪ Pure noise - no image left!"
            status_color = "#6b7280"
        
        self.timestep_display.value = f"""
        <div style="display: flex; justify-content: center; gap: 30px; margin: 15px 0; font-family: 'Inter', sans-serif;">
            <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        color: white; padding: 15px 30px; border-radius: 12px; text-align: center;">
                <div style="font-size: 12px; opacity: 0.9;">Timestep</div>
                <div style="font-size: 32px; font-weight: 700;">t = {timestep}</div>
            </div>
            <div style="background: linear-gradient(135deg, {status_color}dd 0%, {status_color}99 100%);
                        color: white; padding: 15px 30px; border-radius: 12px; text-align: center; min-width: 200px;">
                <div style="font-size: 14px; font-weight: 600;">{status}</div>
            </div>
        </div>
        """
        
        with self.plot_output:
            clear_output(wait=True)
            
            # Generate noisy versions at different stages
            np.random.seed(42)  # Consistent noise
            
            stages = [0, 0.25, 0.5, 0.75, 1.0]
            current_idx = min(4, int(noise_level * 4))
            
            fig = make_subplots(
                rows=1, cols=5,
                subplot_titles=[f't={int(s*1000)}' for s in stages],
                horizontal_spacing=0.02
            )
            
            for i, stage in enumerate(stages):
                np.random.seed(42 + i)
                noisy_img = self.add_noise(self.original_image, stage * 1.5)
                
                # Highlight current position
                opacity = 1.0 if i == current_idx else 0.4
                
                fig.add_trace(go.Heatmap(
                    z=noisy_img[::-1],
                    colorscale='Greys',
                    showscale=False,
                    opacity=opacity,
                    hoverinfo='skip'
                ), row=1, col=i+1)
                
                # Border for current
                if i == current_idx:
                    fig.add_shape(
                        type='rect',
                        x0=-0.5, y0=-0.5, x1=31.5, y1=31.5,
                        line=dict(color='#e73c7e', width=4),
                        row=1, col=i+1
                    )
            
            # Arrow showing direction
            fig.add_annotation(
                x=0.5, y=-0.12,
                xref='paper', yref='paper',
                text='<b>Forward Process: Image → Noise (Training: AI observes this)</b>',
                showarrow=False,
                font=dict(size=13, color='#e73c7e')
            )
            
            fig.update_layout(
                height=300,
                width=1000,
                showlegend=False,
                paper_bgcolor='white',
                margin=dict(l=20, r=20, t=50, b=60)
            )
            
            for i in range(1, 6):
                fig.update_xaxes(visible=False, row=1, col=i)
                fig.update_yaxes(visible=False, row=1, col=i)
            
            fig.show()

# Create the forward diffusion visualizer
forward_viz = DiffusionProcessVisualizer()

In [ ]:
# =============================================================================
# PART 2: THE REVERSE PROCESS - GENERATING IMAGES
# =============================================================================

display(HTML("""
<div class="magic-box">
    <h2 style="margin-top:0;">✨ The Magic: Reversing the Noise</h2>
    <p>Here's the brilliant insight: if we can learn to predict and remove noise...</p>
    <p style="font-size: 1.3em; margin: 15px 0;"><strong>...we can start with PURE NOISE and gradually create an image!</strong></p>
    <p style="opacity: 0.9;">The AI learns: "Given this noisy image, what did the original look like?"</p>
</div>
"""))

class ReverseProcessVisualizer:
    def __init__(self):
        self.create_sample_image()
        self.create_interface()
    
    def create_sample_image(self):
        """Create the target image"""
        img = np.zeros((32, 32))
        y, x = np.ogrid[:32, :32]
        
        # Simple shape (star-like)
        center = (16, 16)
        for angle in np.linspace(0, 2*np.pi, 6)[:-1]:
            for r in range(12):
                px = int(center[0] + r * np.cos(angle))
                py = int(center[1] + r * np.sin(angle))
                if 0 <= px < 32 and 0 <= py < 32:
                    img[py, px] = 0.9
        
        # Center
        center_mask = (x - 16)**2 + (y - 16)**2 <= 4**2
        img[center_mask] = 1.0
        
        self.target_image = img
    
    def generate_denoising_sequence(self, n_steps=50):
        """Generate a sequence from noise to image"""
        np.random.seed(123)
        
        # Start with noise
        noise = np.random.randn(32, 32) * 0.5 + 0.5
        noise = np.clip(noise, 0, 1)
        
        sequence = [noise]
        
        for i in range(1, n_steps + 1):
            t = i / n_steps
            # Interpolate from noise to target (simplified diffusion)
            current = (1 - t) * noise + t * self.target_image
            # Add decreasing noise
            current += np.random.randn(32, 32) * 0.3 * (1 - t)
            current = np.clip(current, 0, 1)
            sequence.append(current)
        
        return sequence
    
    def create_interface(self):
        self.header = widgets.HTML("""
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #e73c7e 100%);
                    color: white; padding: 20px; border-radius: 16px; text-align: center;
                    margin: 20px 0; font-family: 'Inter', sans-serif;
                    box-shadow: 0 8px 30px rgba(102, 126, 234, 0.3);">
            <h3 style="margin: 0;">🎨 Reverse Diffusion: Watch an Image Emerge from Noise!</h3>
            <p style="margin: 8px 0 0 0; font-size: 14px; opacity: 0.95;">
                This is how AI generates images - step by step denoising
            </p>
        </div>
        """)
        
        self.generate_btn = widgets.Button(
            description='✨ Generate Image',
            button_style='success',
            layout=widgets.Layout(width='200px', height='45px')
        )
        
        self.speed_slider = widgets.IntSlider(
            value=50, min=10, max=100, step=10,
            description='Speed (ms):',
            style={'description_width': '80px'},
            layout=widgets.Layout(width='300px')
        )
        
        self.progress_display = widgets.HTML()
        self.plot_output = widgets.Output()
        
        self.generate_btn.on_click(self.run_generation)
        
        display(self.header)
        display(widgets.HBox([self.generate_btn, self.speed_slider], 
                            layout=widgets.Layout(justify_content='center', gap='20px')))
        display(self.progress_display)
        display(self.plot_output)
        
        self.show_static_comparison()
    
    def show_static_comparison(self):
        """Show the before/after comparison"""
        sequence = self.generate_denoising_sequence(20)
        
        with self.plot_output:
            clear_output(wait=True)
            
            fig = make_subplots(
                rows=1, cols=6,
                subplot_titles=['t=1000<br>(Noise)', 't=800', 't=600', 't=400', 't=200', 't=0<br>(Image!)'],
                horizontal_spacing=0.02
            )
            
            indices = [0, 4, 8, 12, 16, 20]
            colors = ['#ef4444', '#f59e0b', '#eab308', '#22c55e', '#10b981', '#06b6d4']
            
            for i, (idx, color) in enumerate(zip(indices, colors)):
                fig.add_trace(go.Heatmap(
                    z=sequence[idx][::-1],
                    colorscale='Greys',
                    showscale=False,
                    hoverinfo='skip'
                ), row=1, col=i+1)
                
                fig.add_shape(
                    type='rect',
                    x0=-0.5, y0=-0.5, x1=31.5, y1=31.5,
                    line=dict(color=color, width=3),
                    row=1, col=i+1
                )
            
            fig.add_annotation(
                x=0.5, y=-0.15,
                xref='paper', yref='paper',
                text='<b>← Reverse Process: Noise → Image (Generation: AI creates new images!)</b>',
                showarrow=False,
                font=dict(size=13, color='#667eea')
            )
            
            fig.update_layout(
                height=280,
                width=1000,
                showlegend=False,
                paper_bgcolor='white',
                margin=dict(l=20, r=20, t=60, b=70)
            )
            
            for i in range(1, 7):
                fig.update_xaxes(visible=False, row=1, col=i)
                fig.update_yaxes(visible=False, row=1, col=i)
            
            fig.show()
    
    def run_generation(self, b):
        """Animate the generation process"""
        import time
        
        sequence = self.generate_denoising_sequence(30)
        delay = self.speed_slider.value / 1000
        
        for i, img in enumerate(sequence):
            progress = i / (len(sequence) - 1)
            timestep = 1000 - int(progress * 1000)
            
            self.progress_display.value = f"""
            <div style="display: flex; justify-content: center; gap: 20px; margin: 10px 0;">
                <div style="background: linear-gradient(90deg, #667eea {progress*100}%, #e5e7eb {progress*100}%);
                            padding: 10px 30px; border-radius: 20px; min-width: 300px;">
                    <div style="color: white; font-weight: 600; text-shadow: 1px 1px 2px rgba(0,0,0,0.3);">
                        Denoising: t = {timestep} → {int(progress*100)}% complete
                    </div>
                </div>
            </div>
            """
            
            with self.plot_output:
                clear_output(wait=True)
                
                fig = go.Figure()
                
                fig.add_trace(go.Heatmap(
                    z=img[::-1],
                    colorscale='Greys',
                    showscale=False
                ))
                
                # Glow effect based on progress
                glow_color = f'rgba(231, 60, 126, {0.3 + progress * 0.5})'
                
                fig.update_layout(
                    height=350,
                    width=400,
                    xaxis=dict(visible=False),
                    yaxis=dict(visible=False),
                    paper_bgcolor='white',
                    margin=dict(l=20, r=20, t=20, b=20),
                    title=dict(
                        text=f'<b>Step {i+1}/{len(sequence)}</b>',
                        x=0.5,
                        font=dict(size=14, color='#667eea')
                    )
                )
                
                fig.show()
            
            time.sleep(delay)
        
        self.progress_display.value = """
        <div style="text-align: center; margin: 15px 0;">
            <span style="background: linear-gradient(135deg, #22c55e 0%, #10b981 100%);
                        color: white; padding: 12px 30px; border-radius: 25px; font-weight: 600;
                        box-shadow: 0 4px 15px rgba(34, 197, 94, 0.3);">
                ✅ Generation Complete! Image created from pure noise!
            </span>
        </div>
        """

# Create reverse process visualizer
reverse_viz = ReverseProcessVisualizer()

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# =============================================================================
# PART 3: HOW TEXT GUIDES GENERATION - INTERACTIVE VISUALIZATION 
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.io as pio
pio.renderers.default = "notebook"


class TextToImagePipelineInteractive:
    """Interactive visualization of the text-to-image diffusion pipeline - Simplified"""
    
    def __init__(self):
        # Define colors
        self.colors = {
            'text': '#f59e0b',
            'encoder': '#667eea',
            'embedding': '#8b5cf6',
            'noise': '#94a3b8',
            'denoiser': '#e73c7e',
            'attention': '#06b6d4',
            'output': '#22c55e',
            'image': '#10b981',
            'highlight': '#FFD700',
            'inactive': '#E8E8E8'
        }
        
        # Simplified steps with concise descriptions
        self.steps = [
            {
                'name': 'Overview',
                'stage': 'all',
                'highlight': [],
                'description': """
                <h2>🎨 Text-to-Image: The Big Picture</h2>
                <p><b>How does AI turn words into pictures?</b></p>
                <div style="background: #fff7ed; padding: 12px; border-radius: 8px; border-left: 4px solid #f59e0b; margin: 10px 0;">
                    <b>Core Idea:</b> Start with noise, gradually remove it while following your text description!
                </div>
                <p><b>The 5 Key Steps:</b></p>
                <ol>
                    <li>You write a text description</li>
                    <li>AI converts text to numbers (embedding)</li>
                    <li>Start with random noise</li>
                    <li>Repeatedly denoise, guided by text</li>
                    <li>Image emerges!</li>
                </ol>
                <p>👉 Click <b>Next</b> to explore each step!</p>
                """
            },
            {
                'name': 'Step 1: Text Prompt',
                'stage': 'text',
                'highlight': [(2.5, 7)],
                'description': """
                <h2>💬 Your Text Prompt</h2>
                <p>Everything starts with your description!</p>
                <div style="background: #fef3c7; padding: 12px; border-radius: 8px; font-family: monospace; margin: 10px 0;">
                    "A cat wearing sunglasses"
                </div>
                <p><b>Better prompts = Better images:</b></p>
                <ul>
                    <li>🎨 Describe the subject</li>
                    <li>🌅 Add environment/setting</li>
                    <li>📷 Specify style (photo, painting, etc.)</li>
                </ul>
                """
            },
            {
                'name': 'Step 2: Text Encoder',
                'stage': 'encoder',
                'highlight': [(7, 7)],
                'description': """
                <h2>🔤 Text Encoder</h2>
                <p>Computers need numbers, not words!</p>
                <p>The encoder converts your text into a list of numbers that capture <b>meaning</b>.</p>
                <div style="background: #e0e7ff; padding: 12px; border-radius: 8px; margin: 10px 0;">
                    "cat wearing sunglasses"<br>
                    ↓<br>
                    [0.23, -0.45, 0.87, ...] <i>(768 numbers!)</i>
                </div>
                <p>Similar concepts get similar numbers!</p>
                """
            },
            {
                'name': 'Step 3: Random Noise',
                'stage': 'noise',
                'highlight': [(2.5, 3)],
                'description': """
                <h2>📡 Start with Noise</h2>
                <p>Here's the surprising part—we begin with <b>pure static!</b></p>
                <div style="background: #f1f5f9; padding: 12px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #64748b;">
                    Think of it like a TV with no signal—just random dots.
                </div>
                <p><b>Why noise?</b></p>
                <p>The noise contains all possible images! Your text guides which one gets revealed.</p>
                <p><b>The "seed":</b> Same seed = same starting noise = reproducible results!</p>
                """
            },
            {
                'name': 'Step 4: Guided Denoising',
                'stage': 'denoiser',
                'highlight': [(7, 3)],
                'description': """
                <h2>🔄 Guided Denoising</h2>
                <p>This is where the magic happens!</p>
                <p>The AI looks at:</p>
                <ul>
                    <li>📡 The current noisy image</li>
                    <li>💬 Your text embedding</li>
                </ul>
                <p>And predicts: <i>"What noise should I remove?"</i></p>
                <div style="background: #fdf2f8; padding: 12px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #e73c7e;">
                    <b>Key:</b> The text tells the AI what to create while removing noise!
                </div>
                <p>This repeats <b>20-50 times</b>, getting cleaner each step.</p>
                """
            },
            {
                'name': 'Step 5: Final Image',
                'stage': 'image',
                'highlight': [(11.5, 3)],
                'description': """
                <h2>🖼️ Your Image Appears!</h2>
                <p>After many denoising steps, your image emerges!</p>
                <div style="background: #d1fae5; padding: 12px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #10b981;">
                    <b>What happened:</b><br>
                    💬 Text → 🔢 Numbers → 📡 Noise → 🔄 Denoise × 50 → 🖼️ Image!
                </div>
                <p><b>Fun facts:</b></p>
                <ul>
                    <li>🎲 Different seed = different variation</li>
                    <li>✏️ Change one word = very different image</li>
                    <li>✨ AI can combine concepts it's never seen together!</li>
                </ul>
                """
            }
        ]
        
        self.current_step = 0
        self.setup_widgets()
    
    def draw_pipeline(self, step_index):
        """Draw the simplified pipeline with larger elements"""
        fig, ax = plt.subplots(1, 1, figsize=(12, 7))
        ax.set_xlim(0, 14)
        ax.set_ylim(0, 10)
        ax.axis('off')
        
        current = self.steps[step_index]
        highlight_coords = current.get('highlight', [])
        
        # Title
        ax.text(7, 9.2, '🎨 Text-to-Image Pipeline',
                fontsize=20, fontweight='bold', ha='center', color='#2C3E50')
        
        def get_color(x, y, default_color):
            if current['stage'] == 'all':
                return default_color, 0.9
            elif any(abs(x - hx) < 0.5 and abs(y - hy) < 0.5 for hx, hy in highlight_coords):
                return self.colors['highlight'], 1.0
            else:
                return self.colors['inactive'], 0.35
        
        def draw_box(x, y, w, h, text, default_color, fontsize=11):
            color, alpha = get_color(x, y, default_color)
            box = FancyBboxPatch(
                (x - w/2, y - h/2), w, h,
                boxstyle="round,pad=0.15",
                facecolor=color, edgecolor='#2C3E50', linewidth=2.5, alpha=alpha
            )
            ax.add_patch(box)
            ax.text(x, y, text, ha='center', va='center',
                    fontsize=fontsize, fontweight='bold', color='#2C3E50')
        
        # ========== TOP ROW: TEXT PROCESSING ==========
        ax.text(4.75, 8.2, 'TEXT PROCESSING', fontsize=12, fontweight='bold', 
                ha='center', color='#555', alpha=0.7)
        
        # Text Prompt (bigger box)
        draw_box(2.5, 7, 3.2, 1.5, '💬 Text Prompt\n"A cat wearing\nsunglasses"', 
                 self.colors['text'], fontsize=11)
        
        # Text Encoder
        draw_box(7, 7, 2.8, 1.5, '🔤 Text\nEncoder', 
                 self.colors['encoder'], fontsize=12)
        
        # Text Embedding
        draw_box(11.5, 7, 2.8, 1.5, '📊 Text\nEmbedding', 
                 self.colors['embedding'], fontsize=12)
        
        # ========== BOTTOM ROW: DIFFUSION ==========
        ax.text(7, 4.3, 'DIFFUSION PROCESS', fontsize=12, fontweight='bold', 
                ha='center', color='#555', alpha=0.7)
        
        # Random Noise
        draw_box(2.5, 3, 2.8, 1.5, '📡 Random\nNoise', 
                 self.colors['noise'], fontsize=12)
        
        # Denoiser (repeated)
        draw_box(7, 3, 3.2, 1.5, '🔄 Denoiser\n(guided by text)', 
                 self.colors['denoiser'], fontsize=11)
        
        # Final Image
        draw_box(11.5, 3, 2.8, 1.5, '🖼️ Final\nImage!', 
                 self.colors['image'], fontsize=12)
        
        # ========== ARROWS ==========
        arrow_style = dict(arrowstyle='->,head_width=0.4,head_length=0.3', 
                          linewidth=3, color='#666')
        
        # Text flow arrows
        active_text = current['stage'] in ['all', 'text', 'encoder']
        alpha_text = 0.8 if active_text else 0.25
        ax.annotate('', xy=(5.5, 7), xytext=(4.2, 7),
                   arrowprops=dict(**arrow_style, alpha=alpha_text))
        
        active_enc = current['stage'] in ['all', 'encoder']
        alpha_enc = 0.8 if active_enc else 0.25
        ax.annotate('', xy=(10, 7), xytext=(8.5, 7),
                   arrowprops=dict(**arrow_style, alpha=alpha_enc))
        
        # Embedding to Denoiser (curved down)
        active_emb = current['stage'] in ['all', 'encoder', 'denoiser']
        alpha_emb = 0.8 if active_emb else 0.25
        arrow = FancyArrowPatch(
            (11.5, 6.2), (8.2, 3.8),
            connectionstyle="arc3,rad=0.25",
            arrowstyle='->,head_width=0.4,head_length=0.3',
            linewidth=3, color='#8b5cf6', alpha=alpha_emb
        )
        ax.add_patch(arrow)
        
        # Label for guidance arrow
        if current['stage'] in ['all', 'denoiser']:
            ax.text(10.3, 5.2, 'guides', fontsize=10, fontweight='bold', 
                    color='#8b5cf6', alpha=0.9, rotation=-35)
        
        # Diffusion flow arrows
        active_noise = current['stage'] in ['all', 'noise', 'denoiser']
        alpha_noise = 0.8 if active_noise else 0.25
        ax.annotate('', xy=(5.3, 3), xytext=(4, 3),
                   arrowprops=dict(**arrow_style, alpha=alpha_noise))
        
        active_out = current['stage'] in ['all', 'denoiser', 'image']
        alpha_out = 0.8 if active_out else 0.25
        ax.annotate('', xy=(10, 3), xytext=(8.7, 3),
                   arrowprops=dict(**arrow_style, alpha=alpha_out))
        
        # ========== ITERATION LOOP ==========
        if current['stage'] in ['all', 'denoiser']:
            loop_alpha = 0.7
        else:
            loop_alpha = 0.2
        
        # Loop arrow under denoiser
        loop = FancyArrowPatch(
            (8.2, 2.1), (5.8, 2.1),
            connectionstyle="arc3,rad=-0.4",
            arrowstyle='->,head_width=0.3,head_length=0.25',
            linewidth=2.5, color='#22c55e', alpha=loop_alpha,
            linestyle='--'
        )
        ax.add_patch(loop)
        ax.text(7, 1.3, '🔄 Repeat 20-50×', fontsize=11, fontweight='bold', 
                ha='center', color='#22c55e', alpha=loop_alpha)
        
        # ========== STEP NUMBERS ==========
        step_positions = [
            (2.5, 7, "1"),
            (7, 7, "2"),
            (2.5, 3, "3"),
            (7, 3, "4"),
            (11.5, 3, "5"),
        ]
        
        for x, y, num in step_positions:
            step_num = int(num)
            is_active = (current['stage'] == 'all' or 
                        (step_index > 0 and step_index == step_num))
            alpha = 1.0 if is_active else 0.3
            
            circle = Circle((x + 1.2, y + 0.6), 0.28, 
                           facecolor='#e73c7e' if is_active else '#ccc',
                           edgecolor='white', linewidth=2, alpha=alpha)
            ax.add_patch(circle)
            ax.text(x + 1.2, y + 0.6, num, ha='center', va='center', 
                   fontsize=10, fontweight='bold', color='white')
        
        plt.tight_layout()
        return fig
    
    def setup_widgets(self):
        """Setup interactive widgets"""
        self.prev_button = widgets.Button(
            description='◀ Previous',
            button_style='primary',
            layout=widgets.Layout(width='110px', height='38px')
        )
        self.next_button = widgets.Button(
            description='Next ▶',
            button_style='primary',
            layout=widgets.Layout(width='110px', height='38px')
        )
        self.reset_button = widgets.Button(
            description='↺ Reset',
            button_style='info',
            layout=widgets.Layout(width='110px', height='38px')
        )
        
        self.step_label = widgets.HTML(
            value=f"<h3 style='text-align: center; color: #2C3E50; margin: 5px 0;'>{self.steps[0]['name']}</h3>"
        )
        
        self.progress = widgets.IntProgress(
            value=1, min=1, max=len(self.steps),
            description='Step:',
            bar_style='success',
            style={'bar_color': '#e73c7e'},
            layout=widgets.Layout(width='100%')
        )
        
        self.description_output = widgets.HTML(
            value=self.steps[0]['description'],
            layout=widgets.Layout(
                width='380px',
                height='420px',
                padding='10px',
                border='2px solid #e0e0e0',
                border_radius='12px',
                overflow_y='auto'
            )
        )
        
        self.figure_output = widgets.Output(
            layout=widgets.Layout(width='750px', height='450px')
        )
        
        def on_prev_click(b):
            if self.current_step > 0:
                self.current_step -= 1
                self.update_display()
        
        def on_next_click(b):
            if self.current_step < len(self.steps) - 1:
                self.current_step += 1
                self.update_display()
        
        def on_reset_click(b):
            self.current_step = 0
            self.update_display()
        
        self.prev_button.on_click(on_prev_click)
        self.next_button.on_click(on_next_click)
        self.reset_button.on_click(on_reset_click)
        
        self.update_display()
    
    def update_display(self):
        """Update the display"""
        self.step_label.value = f"<h3 style='text-align: center; color: #2C3E50; margin: 5px 0;'>{self.steps[self.current_step]['name']}</h3>"
        self.progress.value = self.current_step + 1
        
        self.description_output.value = f"""
        <div style='
            background: linear-gradient(135deg, #fdf2f8 0%, #dbeafe 100%);
            padding: 15px;
            border-radius: 10px;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
            height: 100%;
            box-sizing: border-box;
            font-size: 14px;
        '>
            {self.steps[self.current_step]['description']}
        </div>
        """
        
        self.prev_button.disabled = (self.current_step == 0)
        self.next_button.disabled = (self.current_step == len(self.steps) - 1)
        
        with self.figure_output:
            clear_output(wait=True)
            fig = self.draw_pipeline(self.current_step)
            plt.show()
    
    def display(self):
        """Display the widget"""
        title = widgets.HTML(value="""
            <div style='background: linear-gradient(-45deg, #ee7752, #e73c7e, #23a6d5, #23d5ab);
                        background-size: 400% 400%; color: white; padding: 15px; 
                        border-radius: 12px; text-align: center; margin-bottom: 10px;
                        box-shadow: 0 4px 15px rgba(231, 60, 126, 0.3);'>
                <h2 style='margin: 0; font-size: 1.4em;'>🎨 Interactive Text-to-Image Pipeline</h2>
                <p style='margin: 5px 0 0 0; font-size: 0.95em; opacity: 0.95;'>
                    How AI transforms words into images
                </p>
            </div>
        """)
        
        controls = widgets.HBox([
            self.prev_button, self.reset_button, self.next_button
        ], layout=widgets.Layout(justify_content='center', margin='8px 0'))
        
        content = widgets.HBox([
            self.figure_output,
            self.description_output
        ], layout=widgets.Layout(gap='10px'))
        
        main = widgets.VBox([
            title, self.step_label, self.progress, controls, content
        ], layout=widgets.Layout(width='100%', padding='5px'))
        
        display(main)


# Create and display
pipeline = TextToImagePipelineInteractive()
pipeline.display()
